In [3]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[1]").appName("my_learning_of_ml_spark_scaling").getOrCreate()

In [4]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [5]:
datalist = [
    ('scaling', 2000),
    ('ml', 3000),
    ('pyspark', 4000)
]
rdd = spark.sparkContext.parallelize(datalist)

In [6]:
rdd.take(1)

[('scaling', 2000)]

In [7]:
rdd.take(3)

[('scaling', 2000), ('ml', 3000), ('pyspark', 4000)]

In [8]:
rdd.take(4)

[('scaling', 2000), ('ml', 3000), ('pyspark', 4000)]

In [9]:
rdd.collect()

[('scaling', 2000), ('ml', 3000), ('pyspark', 4000)]

In [10]:
rdd.count()

3

In [11]:
rdd.countByKey()

defaultdict(int, {'scaling': 1, 'ml': 1, 'pyspark': 1})

In [12]:
rdd.countByValue()

defaultdict(int, {('scaling', 2000): 1, ('ml', 3000): 1, ('pyspark', 4000): 1})

In [13]:
rdd.first()

('scaling', 2000)

In [14]:
rdd.take(1)[0]

('scaling', 2000)

In [15]:
rdd.takeOrdered(2)

[('ml', 3000), ('pyspark', 4000)]

In [16]:
data = [('Adi','','Polak','1991-04-01','M',3000),
  ('Michael','Smith','','2000-05-19','M',4000),
  ('Robert','','Jhonie','1978-09-05','M',4000),
  ('Maria','Anne','Swiss','1967-12-01','F',4000),
  ('Jen','Condo','Brown','1980-02-17','F',-1)
]
columns = ["firstname","middlename","lastname","dob","gender","salary"]
df = spark.createDataFrame(data=data, schema=columns)

In [17]:
df.collect()

[Row(firstname='Adi', middlename='', lastname='Polak', dob='1991-04-01', gender='M', salary=3000),
 Row(firstname='Michael', middlename='Smith', lastname='', dob='2000-05-19', gender='M', salary=4000),
 Row(firstname='Robert', middlename='', lastname='Jhonie', dob='1978-09-05', gender='M', salary=4000),
 Row(firstname='Maria', middlename='Anne', lastname='Swiss', dob='1967-12-01', gender='F', salary=4000),
 Row(firstname='Jen', middlename='Condo', lastname='Brown', dob='1980-02-17', gender='F', salary=-1)]

In [18]:
df.printSchema()

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)



In [19]:
df.show()

+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|       dob|gender|salary|
+---------+----------+--------+----------+------+------+
|      Adi|          |   Polak|1991-04-01|     M|  3000|
|  Michael|     Smith|        |2000-05-19|     M|  4000|
|   Robert|          |  Jhonie|1978-09-05|     M|  4000|
|    Maria|      Anne|   Swiss|1967-12-01|     F|  4000|
|      Jen|     Condo|   Brown|1980-02-17|     F|    -1|
+---------+----------+--------+----------+------+------+



In [22]:
df2 = rdd.toDF(schema=['a','b'])
df2.printSchema()

root
 |-- a: string (nullable = true)
 |-- b: long (nullable = true)



In [23]:
df2.show(truncate=False)

+-------+----+
|a      |b   |
+-------+----+
|scaling|2000|
|ml     |3000|
|pyspark|4000|
+-------+----+



In [24]:
from pyspark.sql.types import StructType, StructField, StringType

deptSchema = StructType([
    StructField('name', StringType(), True),
    StructField('id', StringType(), True)
])
deptdf1 = spark.createDataFrame(rdd, schema=deptSchema)
deptdf1.printSchema()
deptdf1.show(truncate=False)

root
 |-- name: string (nullable = true)
 |-- id: string (nullable = true)

+-------+----+
|name   |id  |
+-------+----+
|scaling|2000|
|ml     |3000|
|pyspark|4000|
+-------+----+



In [25]:
# temp view
df.createOrReplaceTempView("PERSON_DATA")
df2 = spark.sql("SELECT * FROM PERSON_DATA")
df2.printSchema()
df2.show()

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)

+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|       dob|gender|salary|
+---------+----------+--------+----------+------+------+
|      Adi|          |   Polak|1991-04-01|     M|  3000|
|  Michael|     Smith|        |2000-05-19|     M|  4000|
|   Robert|          |  Jhonie|1978-09-05|     M|  4000|
|    Maria|      Anne|   Swiss|1967-12-01|     F|  4000|
|      Jen|     Condo|   Brown|1980-02-17|     F|    -1|
+---------+----------+--------+----------+------+------+



In [33]:
# group by gender
#df.createTempView("PERSON_DATA_2")
df3 = spark.sql("SELECT gender, count(*) as Count FROM PERSON_DATA GROUP BY gender")
df3.printSchema()
df3.show()

root
 |-- gender: string (nullable = true)
 |-- Count: long (nullable = false)

+------+-----+
|gender|Count|
+------+-----+
|     F|    2|
|     M|    3|
+------+-----+



## Twitter data

In [44]:
df = spark.read.csv('bot_data.csv', header=True)
df.printSchema()
df.show(truncate=True)

root
 |-- id: string (nullable = true)
 |-- id_str: string (nullable = true)
 |-- screen_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- description: string (nullable = true)
 |-- url: string (nullable = true)
 |-- followers_count: string (nullable = true)
 |-- friends_count: string (nullable = true)
 |-- listed_count: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- favourites_count: string (nullable = true)
 |-- verified: string (nullable = true)
 |-- statuses_count: string (nullable = true)
 |-- lang: string (nullable = true)
 |-- status: string (nullable = true)
 |-- default_profile: string (nullable = true)
 |-- default_profile_image: string (nullable = true)
 |-- has_extended_profile: string (nullable = true)
 |-- name: string (nullable = true)
 |-- bot: string (nullable = true)

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------

In [45]:
df.count()

2840

In [46]:
df_new = df.select('bot')

In [47]:
df.limit(2).toPandas()

,id,id_str,screen_name,location,description,url,followers_count,friends_count,listed_count,created_at,favourites_count,verified,statuses_count,lang,status,default_profile,default_profile_image,has_extended_profile,name,bot
0,8.16E+17,"""""""815745789754417152""""""","""""""HoustonPokeMap""""""","""""""Houston","TX""""""","""""""Rare and strong PokŽmon in Houston",TX. See more PokŽmon at https://t.co/dnWuDbFR...,"""""""https://t.co/dnWuDbFRkt""""""",1291,0,10,"""""""Mon Jan 02 02:25:26 +0000 2017""""""",0,FALSE,78554,"""""""en""""""","""{ """"created_at"""": """"Sun Mar 12 15:44:04 ...","""""id"""": 840951532543737900","""""id_str"""": """"840951532543737856""""","""""text"""": """"[Southeast Houston] Chansey ..."
1,4843621225,4843621225,kernyeahx,"Templeville town, MD, USA",From late 2014 Socium Marketplace will make sh...,None,1,349,0,2/1/2016 7:37,38,FALSE,31,en,null,TRUE,FALSE,FALSE,Keri Nelson,1


In [50]:
df_new = df.select('bot')
df_new.limit(2).toPandas()

,bot
0,"""""text"""": """"[Southeast Houston] Chansey ..."
1,1


In [51]:
df.select('bot').limit(2).toPandas()

,bot
0,"""""text"""": """"[Southeast Houston] Chansey ..."
1,1


In [52]:
df.take(2)

[Row(id='8.16E+17', id_str='"""815745789754417152"""', screen_name='"""HoustonPokeMap"""', location='"""Houston', description=' TX"""', url='"""Rare and strong PokŽmon in Houston', followers_count=' TX. See more PokŽmon at https://t.co/dnWuDbFRkt"""', friends_count='"""https://t.co/dnWuDbFRkt"""', listed_count='1291', created_at='0', favourites_count='10', verified='"""Mon Jan 02 02:25:26 +0000 2017"""', statuses_count='0', lang='FALSE', status='78554', default_profile='"""en"""', default_profile_image='"{      ""created_at"": ""Sun Mar 12 15:44:04 +0000 2017""', has_extended_profile='      ""id"": 840951532543737900', name='      ""id_str"": ""840951532543737856""', bot='      ""text"": ""[Southeast Houston] Chansey (F) (IV: 73%) until 11:11:37AM at 2511 Winbern St https://t.co/HYRIyq4mF7 https://t.co/bydOOKsEEI""'),
 Row(id='4843621225', id_str='4843621225', screen_name='kernyeahx', location='Templeville town, MD, USA', description='From late 2014 Socium Marketplace will make shoppin

In [53]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- id_str: string (nullable = true)
 |-- screen_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- description: string (nullable = true)
 |-- url: string (nullable = true)
 |-- followers_count: string (nullable = true)
 |-- friends_count: string (nullable = true)
 |-- listed_count: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- favourites_count: string (nullable = true)
 |-- verified: string (nullable = true)
 |-- statuses_count: string (nullable = true)
 |-- lang: string (nullable = true)
 |-- status: string (nullable = true)
 |-- default_profile: string (nullable = true)
 |-- default_profile_image: string (nullable = true)
 |-- has_extended_profile: string (nullable = true)
 |-- name: string (nullable = true)
 |-- bot: string (nullable = true)



In [54]:
df.limit(25).toPandas()

,id,id_str,screen_name,location,description,url,followers_count,friends_count,listed_count,created_at,favourites_count,verified,statuses_count,lang,status,default_profile,default_profile_image,has_extended_profile,name,bot
0,8.16E+17,"""""""815745789754417152""""""","""""""HoustonPokeMap""""""","""""""Houston","TX""""""","""""""Rare and strong PokŽmon in Houston",TX. See more PokŽmon at https://t.co/dnWuDbFR...,"""""""https://t.co/dnWuDbFRkt""""""",1291,0,10,"""""""Mon Jan 02 02:25:26 +0000 2017""""""",0,FALSE,78554,"""""""en""""""","""{ """"created_at"""": """"Sun Mar 12 15:44:04 ...","""""id"""": 840951532543737900","""""id_str"""": """"840951532543737856""""","""""text"""": """"[Southeast Houston] Chansey ..."
1,4843621225,4843621225,kernyeahx,"Templeville town, MD, USA",From late 2014 Socium Marketplace will make sh...,None,1,349,0,2/1/2016 7:37,38,FALSE,31,en,null,TRUE,FALSE,FALSE,Keri Nelson,1
2,4303727112,4303727112,mattlieberisbot,None,"Inspired by the smart, funny folks at @replyal...",https://t.co/P1e1o0m4KC,1086,0,14,Fri Nov 20 18:53:22 +0000 2015,0,FALSE,713,en,"""{'retweeted': False, 'is_quote_status': False...",'truncated': False,'in_reply_to_user_id': None,'created_at': 'Mon Mar 13 16:00:00 +0000 2017','contributors': None,'in_reply_to_status_id_str': None
3,3063139353,3063139353,sc_papers,None,None,None,33,0,8,2/25/2015 20:11,0,FALSE,676,en,Construction of human anti-tetanus single-chai...,TRUE,TRUE,FALSE,single cell papers,1
4,2955142070,2955142070,lucarivera16,"Dublin, United States",Inspiring cooks everywhere since 1956.,None,11,745,0,1/1/2015 17:44,146,FALSE,185,en,null,FALSE,FALSE,FALSE,lucarivera16,1
5,8.41E+17,8.41E+17,dantheimprover,"Austin, TX",Just a guy trying to do good by telling everyo...,None,1,186,0,13/03/2017 22:53,0,FALSE,11,en,"""Status(_api=<tweepy.api.API object at 0x10192...",'in_reply_to_status_id': None,'in_reply_to_status_id_str': None,'in_reply_to_user_id': None,'in_reply_to_user_id_str': None,'in_reply_to_screen_name': None
6,2482834658,2482834658,_all_of_us_,in a machine.,bot by @rubicon,None,193,0,19,Wed May 07 22:29:25 +0000 2014,0,FALSE,6068,en,"""{u'contributors': None, u'truncated': False, ...",u'retweeted': False,u'coordinates': None,u'entities': {u'symbols': [],u'user_mentions': [],u'hashtags': []
7,3333573622,3333573622,KatamariItems,None,[Bot rolled up by @BeachEpisode] Cataloguing e...,None,8227,2,89,Thu Jun 18 22:07:31 +0000 2015,26,FALSE,2597,en,"""{u'contributors': None, u'truncated': False, ...",u'retweeted': False,u'coordinates': None,u'entities': {u'symbols': [],u'user_mentions': [],u'hashtags': []
8,2996105102,2996105102,AutophagyPapers,None,Twitterbot for #Autophagy papers. Curated by @...,None,275,0,17,1/25/2015 17:34,23,FALSE,9922,en,Feeding Schedule And Proteolysis Regulate Auto...,FALSE,FALSE,FALSE,Autophagy Papers,1
9,3271095818,3271095818,HSC_papers,None,None,None,51,3,9,7/7/2015 15:23,0,FALSE,2515,en,Functional Selectivity in Cytokine Signaling R...,TRUE,FALSE,FALSE,Hematopoiesis,1


In [55]:
df.explain(True)

== Parsed Logical Plan ==
Relation[id#705,id_str#706,screen_name#707,location#708,description#709,url#710,followers_count#711,friends_count#712,listed_count#713,created_at#714,favourites_count#715,verified#716,statuses_count#717,lang#718,status#719,default_profile#720,default_profile_image#721,has_extended_profile#722,name#723,bot#724] csv

== Analyzed Logical Plan ==
id: string, id_str: string, screen_name: string, location: string, description: string, url: string, followers_count: string, friends_count: string, listed_count: string, created_at: string, favourites_count: string, verified: string, statuses_count: string, lang: string, status: string, default_profile: string, default_profile_image: string, has_extended_profile: string, name: string, bot: string
Relation[id#705,id_str#706,screen_name#707,location#708,description#709,url#710,followers_count#711,friends_count#712,listed_count#713,created_at#714,favourites_count#715,verified#716,statuses_count#717,lang#718,status#719,defau